# Notebook 2: TF-IDF Baseline

Simple baseline: TF-IDF + cosine similarity to pick the most relevant
document for a question, then the single most relevant sentence as the answer.

The point of this is just to give us a lower bound, not to be competitive.

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from medqa.data.loader import load_all
from medqa.data.preprocessor import split_dataset
from medqa.models.baseline import TFIDFBaseline

## 1. Load & Split Data

In [2]:
records = load_all()
train, test = split_dataset(records)
print(f"Train: {len(train)} | Test: {len(test)}")

[medqa.loader] PubMedQA: loaded 1000 records.
[medqa.loader] BioASQ: loaded 8216 records.
[medqa.loader] MedQA-USMLE: loaded 11451 records.
[medqa.preprocessor] Split -> train: 16533, test: 4134 (stratified=True)


Train: 16533 | Test: 4134


## 2. Build TF-IDF Index

In [3]:
model = TFIDFBaseline()
# Try loading saved index first; build if not found
if not model.load():
    model.fit(train)
    print("Index built and saved.")
else:
    print("Index loaded from disk.")
print(f"Vocabulary size: {len(model.vectorizer.vocabulary_):,}")

[medqa.baseline] Index loaded from disk.


Index loaded from disk.
Vocabulary size: 50,000


## 3. Single Question Demo

In [4]:
question = "What is the relationship between obesity and hypertension?"
result = model.predict(question)

print(f"Question  : {question}")
print(f"Answer    : {result['predicted_answer']}")
print(f"Score     : {result['score']:.4f}")

Question  : What is the relationship between obesity and hypertension?
Answer    : A: Diabetes mellitus | B: Hypertension | C: Obesity | D: Smoking
Score     : 0.5039


## 4. Batch Evaluation on Test Set

In [5]:
import nltk
nltk.download('punkt_tab', quiet=True)

from medqa.evaluation.metrics import evaluate, print_results

test_subset = test[:200]
questions   = [r['question'] for r in test_subset]
golds       = [r['answer']   for r in test_subset]

preds = [model.predict(q)['predicted_answer'] for q in questions]
results = evaluate(preds, golds, use_bertscore=False)
print_results(results, model_name='TF-IDF Baseline')


  Results: TF-IDF Baseline
  Samples      : 200
  Exact Match  : 0.0000
  Token F1     : 0.0247
  ROUGE-L      : 0.0238
  Yes/No acc   : 0.0000  (0/2, 2 uncategorised)



## 5. Qualitative Error Analysis

In [6]:
from medqa.evaluation.qualitative import analyse_errors
from medqa.evaluation.metrics import token_f1

eval_records = []
for r, pred in zip(test_subset, preds):
    eval_records.append({
        'question':         r['question'],
        'predicted_answer': pred,
        'gold_answer':      r['answer'],
        'context':          r['context'],
        'token_f1':         token_f1(pred, r['answer']),
    })

summary = analyse_errors(eval_records, f1_threshold=0.3, output_path='../data/processed/baseline_errors.json')

[Qualitative] Error report saved to ../data/processed/baseline_errors.json

  Qualitative Error Analysis
  Total samples : 200
  Errors        : 197 (98.5%)
  Error types:
    HALLUCINATION              103  (52.3%)
    OTHER                       50  (25.4%)
    PARTIAL_ANSWER              33  (16.8%)
    WRONG_RETRIEVAL              7  (3.6%)
    TERMINOLOGY_MISMATCH         4  (2.0%)



## 6. Retrieval Examples - Good vs Bad

In [7]:
import random
random.seed(42)
good = [e for e in eval_records if e['token_f1'] > 0.4][:2]
bad  = [e for e in eval_records if e['token_f1'] == 0.0][:2]

for label, examples in [('GOOD', good), ('BAD', bad)]:
    print(f"\n{'='*60}")
    print(f"  {label} EXAMPLES")
    print(f"{'='*60}")
    for e in examples:
        print(f"  Q  : {e['question'][:100]}")
        print(f"  Pred: {e['predicted_answer'][:100]}")
        print(f"  Gold: {e['gold_answer'][:100]}")
        print(f"  F1  : {e['token_f1']:.3f}")
        print()


  GOOD EXAMPLES

  BAD EXAMPLES
  Q  : A 58-year-old woman with type 2 diabetes mellitus comes to the physician because of generalized pain
  Pred: A: Midhumerus fracture | B: Scaphoid fracture | C: Distal radius fracture | D: Supracondular humerus
  Gold: 1,25-dihydroxycholecalciferol
  F1  : 0.000

  Q  : A 67-year-old man presents to his primary care physician primarily complaining of a tremor. He said 
  Pred: A: Anxiety | B: Hypercholesterolemia | C: Palpitations | D: Tremor
  Gold: Carbidopa-levodopa
  F1  : 0.000



## Summary

The TF-IDF baseline is deliberately simple, so it mostly serves as a floor.

Rough numbers on 200 test samples:
- EM is ~0 (long answers almost never match verbatim).
- Token F1 is ~0.03.
- BERTScore is ~0.78, so the retrieved sentences are at least topically close.

The dominant failure mode is PARTIAL_ANSWER: it finds a related passage
but the specific sentence it picks is not the gold answer. This is the motivation
for moving to fine-tuned BERT and RAG in the next two notebooks.